# Agentic keyword-assignment workflow

This workflow uses the model as the planner:
1. retrieve relevant examples
2. ask the model for candidate keywords
3. map each candidate to the canonical vocabulary
4. return the final mapped result as a DataFrame

In [ ]:
import json
import re
import pandas as pd
import requests
import weaviate

from src.retrieve import ExampleRetriever
from src.mapping import Mapping


def call_model(messages, model="local-model"):
    """
    OpenAI-compatible vLLM call.
    Replace model name if needed.
    """
    payload = {
        "model": model,
        "messages": messages,
        "temperature": 0.0,
        "max_tokens": 256,
    }
    r = requests.post(
        "http://127.0.0.1:9513/v1/chat/completions",
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"]


def retrieve_examples(text, collection_name="title_train", n_examples=5):
    df = pd.DataFrame(
        {
            "text": [text],
            "doc_id": [1],
            "label_ids": [[1]],
            "label_texts": [[text]],
        }
    )

    retriever = ExampleRetriever(
        input_text_data=df,
        output_file=None,
        n_examples=n_examples,
        collection_name=collection_name,
        host="8090",
        debug=False,
    )
    return retriever.retrieve_examples()


def map_candidate(candidate, collection_name="vocab", host="8090"):
    client = weaviate.connect_to_local(port=8087)
    try:
        mapper = Mapping(
            hyperparameters={"host": host, "search": "vector", "use_phrase": False},
            collection_name=collection_name,
            phrase=None,
            debug=False,
            db_connection=client,
        )
        return mapper.query_vector_database(candidate)
    finally:
        client.close()


def parse_keywords(raw_text):
    try:
        obj = json.loads(raw_text)
        if isinstance(obj, dict) and "keywords" in obj:
            return obj["keywords"]
        if isinstance(obj, list):
            return obj
    except Exception:
        pass

    matches = re.findall(r'"([^"]+)"', raw_text)
    if matches:
        return matches

    return []


def agent_solve(text):
    examples = retrieve_examples(text, collection_name="title_train", n_examples=5)

    example_block = "\n".join(
        [
            f"- {row['prompt_text']} -> {row['prompt_label_texts']}"
            for _, row in examples.iterrows()
        ]
    )

    system_prompt = (
        "You are a subject-indexing assistant. "
        "Use the retrieved examples as style and domain guidance. "
        "Return valid JSON with a top-level 'keywords' list of candidate subject terms."
    )

    user_prompt = f"""
    Text:
    {text}

    Relevant examples:
    {example_block}

    Return only JSON in the form:
    {{"keywords": ["keyword1", "keyword2", ...]}}
    """

    raw = call_model(
        [{"role": "system", "content": system_prompt},
         {"role": "user", "content": user_prompt}]
    )

    candidates = parse_keywords(raw)
    mapped = []
    for cand in candidates:
        mapped_result = map_candidate(cand, collection_name="vocab", host="8090")
        mapped.append(next(iter(mapped_result.values())))

    # flatten into a DataFrame
    rows = []
    for item in mapped:
        if isinstance(item, dict):
            rows.append({"candidate": list(item.keys())[0], **list(item.values())[0]})

    return {
        "text": text,
        "examples": examples.to_dict(orient="records"),
        "candidates": candidates,
        "mapped_rows": rows,
        "df": pd.DataFrame(rows),
    }


result = agent_solve("Ein spannendes Buch über mathematische Optimierung und Gradientenverfahren")
result["df"]